# Preprocessing for Clustering

This notebook creates the first model-ready preprocessing output from the customer-level feature table. It does not run clustering or modelling.

## Imports

Use the reusable preprocessing functions from `src/preprocessing.py`.

In [12]:
import sys

import pandas as pd

sys.path.append("../src")

from preprocessing import clean_feature_values, preprocess_for_clustering, select_model_features

## Load Customer Features

Load the combined customer-level feature table created during feature engineering.

In [13]:
customer_features = pd.read_csv("../data/processed/customer_features.csv")
customer_features.head()

,customer_id,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,year_first_transaction,percentage_of_products_bought_promotion,typical_hour,...,share_petfood,degree_level,basket_count,avg_basket_size,max_basket_size,min_basket_size,total_items_bought_in_baskets,distinct_products_in_baskets,avg_distinct_products_per_basket,most_frequent_product
0,3,female,1.0,1.0,1.0,3.0,189.0,2020.0,0.631599,NaN,...,0.020656,Bsc,2,10.5,11,10,21,21,10.5,asparagus
1,4,female,1.0,0.0,0.0,2.0,130.0,2013.0,0.149890,NaN,...,0.032867,Bsc,2,12.0,12,12,24,20,12.0,airpods
2,5,male,0.0,0.0,NaN,2.0,81.0,2005.0,0.069126,11.0,...,0.014277,Msc,1,8.0,8,8,8,8,8.0,bramble
3,7,male,0.0,0.0,2.0,1.0,92.0,2021.0,0.253609,18.0,...,0.012306,Unknown,1,4.0,4,4,4,4,4.0,cream
4,8,male,0.0,0.0,3.0,1.0,6.0,2021.0,0.186569,17.0,...,0.017095,Unknown,0,0.0,0,0,0,0,0.0,No Basket


## Initial Checks

Check shape, duplicate customer IDs, and missing values before preprocessing.

In [14]:
print(f"Rows before preprocessing: {customer_features.shape[0]:,}")
print(f"Columns before preprocessing: {customer_features.shape[1]:,}")
print(f"Duplicated customer_id before preprocessing: {customer_features['customer_id'].duplicated().sum():,}")

Rows before preprocessing: 33,038
Columns before preprocessing: 37
Duplicated customer_id before preprocessing: 0


In [15]:
missing_before = customer_features.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0]

share_fish                                 991
number_complaints                          661
share_meat                                 661
typical_hour                               661
share_videogames                           661
share_vegetables                           661
share_electronics                          661
share_petfood                              661
total_children_home                        656
kids_home                                  330
teens_home                                 330
distinct_stores_visited                    330
share_alcohol_drinks                       330
percentage_of_products_bought_promotion    330
share_hygiene                              330
age                                        165
dtype: int64

## Clean and Select Features

Clean invalid and missing values, then remove non-model columns while keeping `customer_id` for traceability.

In [16]:
cleaned_customer_features = clean_feature_values(customer_features)
selected_customer_features = select_model_features(cleaned_customer_features)

removed_columns = [
    column for column in customer_features.columns
    if column not in selected_customer_features.columns
]

removed_columns

['most_frequent_product']

In [17]:
missing_after_cleaning = selected_customer_features.isna().sum().sort_values(ascending=False)
missing_after_cleaning[missing_after_cleaning > 0]

Series([], dtype: int64)

## Preprocess Model Features

One-hot encode categorical model columns and scale numeric model features with `StandardScaler`. `customer_id` is preserved and not used as a model feature.

In [18]:
customer_features_model = preprocess_for_clustering(customer_features)
customer_features_model.head()

,customer_id,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,year_first_transaction,percentage_of_products_bought_promotion,typical_hour,latitude,...,min_basket_size,total_items_bought_in_baskets,distinct_products_in_baskets,avg_distinct_products_per_basket,customer_gender_female,customer_gender_male,degree_level_Bsc,degree_level_Msc,degree_level_Phd,degree_level_Unknown
0,3,-0.100447,0.104470,0.076516,-0.099810,0.378445,0.931644,1.131040,-0.134436,1.988434,...,1.113774,-0.266698,-0.136197,0.612862,1,0,1,0,0,0
1,4,-0.100447,-0.939218,-1.052532,-0.700126,-0.178573,-0.459419,-0.643434,-0.134436,0.089678,...,1.623716,-0.159322,-0.185561,0.976777,1,0,1,0,0,0
2,5,-0.974216,-0.939218,0.076516,-0.700126,-0.641180,-2.049206,-0.940948,-0.342479,1.377233,...,0.603833,-0.731995,-0.777931,0.006336,0,1,0,1,0,0
3,7,-0.974216,-0.939218,1.205564,-1.300442,-0.537330,1.130368,-0.261363,1.113819,-0.450951,...,-0.416050,-0.875163,-0.975387,-0.964104,0,1,0,0,0,1
4,8,-0.974216,-0.939218,2.334612,-1.300442,-1.349253,1.130368,-0.508321,0.905776,-0.738851,...,-1.435934,-1.018331,-1.172844,-1.934545,0,1,0,0,0,1


## Final Validation

Confirm the preprocessed output keeps all customers, has no duplicate IDs, and has no missing values in model features.

In [19]:
preprocessing_validation = pd.DataFrame(
    {
        "check": [
            "rows before preprocessing",
            "rows after preprocessing",
            "columns before preprocessing",
            "columns after preprocessing",
            "duplicated customer_id after preprocessing",
            "missing values in model features",
        ],
        "value": [
            customer_features.shape[0],
            customer_features_model.shape[0],
            customer_features.shape[1],
            customer_features_model.shape[1],
            customer_features_model["customer_id"].duplicated().sum(),
            customer_features_model.drop(columns=["customer_id"]).isna().sum().sum(),
        ],
    }
)

preprocessing_validation

,check,value
0,rows before preprocessing,33038
1,rows after preprocessing,33038
2,columns before preprocessing,37
3,columns after preprocessing,40
4,duplicated customer_id after preprocessing,0
5,missing values in model features,0


In [20]:
missing_after = customer_features_model.drop(columns=["customer_id"]).isna().sum().sort_values(ascending=False)
missing_after[missing_after > 0]

Series([], dtype: int64)

In [21]:
customer_features_model.columns.to_frame(index=False, name="feature_column")

,feature_column
0,customer_id
1,kids_home
2,teens_home
3,number_complaints
4,distinct_stores_visited
5,lifetime_total_distinct_products
6,year_first_transaction
7,percentage_of_products_bought_promotion
8,typical_hour
9,latitude


## Save Preprocessed Feature Table

Save the preprocessed customer-level feature table. This file keeps `customer_id` for traceability, but clustering should use the remaining feature columns.

In [22]:
customer_features_model.to_csv("../data/processed/customer_features_model.csv", index=False)
print("Saved ../data/processed/customer_features_model.csv")

Saved ../data/processed/customer_features_model.csv


## Preprocessing Notes

- `customer_id` is kept in the final file for traceability, but it is excluded from model feature scaling.
- `most_frequent_product` is removed because it is useful for profiling but not a simple numeric clustering feature.
- Numeric missing values are filled with the median, and categorical missing values are filled with `Unknown`.
- Invalid `percentage_of_products_bought_promotion` values are clipped to the `0` to `100` range.
- `customer_gender` and `degree_level` are one-hot encoded.
- Numeric model features are scaled with `StandardScaler`.
- No clustering or modelling is performed in this notebook.